In [6]:
import pandas as pd
from tqdm import tqdm  # Import tqdm for progress bars
from concurrent.futures import ThreadPoolExecutor, as_completed
import osmnx as ox
from shapely.geometry import box
import geopandas as gpd
import h3

# Function to query landuse for a single area
def query_landuse(geom, tags={"landuse": "commercial"}):
    try:
        gdf = ox.features_from_polygon(geom, tags)
        return gdf[gdf.geometry.type.isin(["Polygon", "MultiPolygon"])]
    except Exception as e:
        return None

# Split the area into smaller pieces
def split_area(geometry, num_splits=4):
    minx, miny, maxx, maxy = geometry.bounds
    dx = (maxx - minx) / num_splits
    dy = (maxy - miny) / num_splits
    pieces = []
    for i in range(num_splits):
        for j in range(num_splits):
            pieces.append(box(minx + i * dx, miny + j * dy, minx + (i + 1) * dx, miny + (j + 1) * dy))
    return pieces

# Parallel query with progress tracking
def query_with_progress_concurrent(boundary_geometry, num_splits=4, max_workers=4):
    # Split the area into pieces
    pieces = split_area(boundary_geometry, num_splits=num_splits)

    # Use ThreadPoolExecutor with progress tracking
    results = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(query_landuse, piece) for piece in pieces]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Querying Sub-areas"):
            try:
                result = future.result()  # Get the result of the completed task
                results.append(result)    # Append the result to the results list
            except Exception as e:
                print(f"Error: {e}")
            
    combined_gdf = gpd.GeoDataFrame(pd.concat(results, ignore_index=True))
    return combined_gdf


In [ ]:


# Get the UK boundary
uk_boundary = ox.geocode_to_gdf("United Kingdom").geometry[0]
# Query retail landuse data in parallel with progress tracking
retail_landuse = query_with_progress_concurrent(uk_boundary, num_splits=10, max_workers=16)

# Clip the data to the UK boundary and remove duplicates
retail_landuse = (
    gpd.clip(retail_landuse, uk_boundary)
    [['geometry', 'landuse', 'name']]
    .drop_duplicates()
)
#save to file (geoparquet)
retail_landuse.to_parquet("../../data/input_files/commercial_landuse.parquet")

Querying Sub-areas: 100%|██████████| 100/100 [00:24<00:00,  4.15it/s]


FileNotFoundError: [Errno 2] Failed to open local file '../data/input_files/commercial_landuse.parquet'. Detail: [errno 2] No such file or directory

In [8]:
retail_landuse = gpd.read_parquet("../../data/input_files/commercial_landuse.parquet")

In [9]:
import h3

# Function to convert polygon or multipolygon to H3 indices
def get_h3_indices(geometry, resolution=13):
    # Check if the geometry is a Polygon or MultiPolygon
    if geometry.geom_type == 'Polygon':
        coords = [(lat, lng) for lng, lat in geometry.exterior.coords]
    elif geometry.geom_type == 'MultiPolygon':
        coords = []
        # Use geoms to iterate over individual polygons in the MultiPolygon
        for poly in geometry.geoms:
            coords.extend([(lat, lng) for lng, lat in poly.exterior.coords])
    else:
        raise ValueError(f"Geometry type {geometry.geom_type} not supported")

    # Create a LatLngPoly object
    poly = h3.LatLngPoly(coords)
    # Convert the polygon to H3 cells at the specified resolution
    return h3.h3shape_to_cells(poly, res=resolution)

# # Apply the function to the GeoDataFrame to get H3 indices at resolution 13
retail_landuse['h3_indices_res13'] = retail_landuse['geometry'].apply(get_h3_indices, resolution=13)
retail_landuse['h3_indices_res11'] = retail_landuse['geometry'].apply(get_h3_indices, resolution=11)

# Now apply the function to convert the H3 indices to resolution 11
#drop rows with empty h3_indices_res13, these are a handful of very small polygons
retail_landuse = retail_landuse[retail_landuse['h3_indices_res13'].apply(len) > 0]
retail_landuse['h3_indices_res11_parents'] = retail_landuse['h3_indices_res13'].apply(lambda h3_cells: [h3.cell_to_parent(cell, 11) for cell in h3_cells])
#explode h3_indices_res11_parents and rename to h3_address
retail_landuse_h3 = retail_landuse.explode('h3_indices_res11_parents').rename(columns={'h3_indices_res11_parents': 'h3_11'}).drop(columns=['h3_indices_res13', 'h3_indices_res11',"name","geometry"])
#drop repeats

retail_landuse_h3 = retail_landuse_h3.drop_duplicates()

#save to file (parquet)
retail_landuse_h3.to_parquet("../../data/input_files/commercial_landuse_h3.parquet")

In [18]:
print(retail_landuse_h3.head())

          landuse            h3_11
17122  commercial  8b194a04d232fff
17122  commercial  8b194a04d389fff
17122  commercial  8b194a04d236fff
17122  commercial  8b194a04d214fff
17122  commercial  8b194a04d38dfff


In [10]:
import geopandas as gpd
new_centres = gpd.read_parquet("../../data/output_files/mergeafterfiltthresh_retailcen_foursquare_mrc_2_mri_10_mrn_10_mtf_6_splitthresh_1500_merge_y/3_afterfill.parquet")

In [11]:
# Step 1: Make a set of all valid h3_11 values from the land use table
landuse_h3_set = set(retail_landuse_h3['h3_11'])

# Step 2: For each network_number, get the set of h3_11 values
network_h3s = new_centres.groupby('network_number')['h3_11'].apply(set)

# Step 3: Find network_numbers where all h3_11s are in the landuse set
valid_networks = network_h3s[network_h3s.apply(lambda s: s.issubset(landuse_h3_set))].index

# Step 4: Filter new_centres to keep only those network_numbers
filtered_centres = new_centres[new_centres['network_number'].isin(valid_networks)].copy()

# Step 5: Count how many such networks there are
num_valid_networks = filtered_centres['network_number'].nunique()

# Optional: add the count as a column if needed (same value for all rows)
filtered_centres['networks_found'] = num_valid_networks

# Final output
print(f"Found {num_valid_networks} networks fully covered by retail land use.")


Found 353 networks fully covered by retail land use.


In [12]:
import importlib.util
from pathlib import Path

# Path to the actual utils.py file
utils_file = Path().resolve().parents[1] / 'src' / 'utils' / 'utils.py'

# Load as module named "ut"
spec = importlib.util.spec_from_file_location("ut", utils_file)
ut = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ut)


# Save DataFrame
ut.to_parquet(filtered_centres, f"commercial_removed.parquet")


/home/ogoodwin/projects/RetailCentres/src/utils/utils.py:14: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df.apply(lambda x: h3.cell_to_boundary(x["h3_11"]), axis=1)
/home/ogoodwin/projects/RetailCentres/src/utils/utils.py:16: UserWarning: Geometry column does not contain geometry.
  df["geometry"] = df["geometry"].apply(lambda x: [(i[1], i[0]) for i in x])
